In this script we show how to use regression LFM for portfolio return stress-testing in a 2-dimensional stock market.

In [1]:
from numpy import array, average, cov, diag, sqrt, sum
from statsmodels.multivariate.factor import Factor
from pandas import read_csv, to_datetime

Load Data

From the database db_stocks_SP500/SPX we upload the S&P 500 index values and compute returns.

In [7]:
# Loading the SPX.csv file into a pandas DataFrame, declaring 'date' as a column of datetimes that could be of mixed format
db_spx = read_csv("data/SPX.csv", parse_dates=['date'], date_format='mixed', index_col=0)

# Filter the SPX prices into a 2-week window (between 16th March -30th March 2012) and flatten into a 1D array or prices
v_spx = db_spx[(db_spx.index >= to_datetime('16-03-2012', dayfirst=True)) &
               (db_spx.index <= to_datetime('30-03-2012', dayfirst=True))].values.reshape(-1)

# Computing daily linear returns of the S&P 500: (price today / price yesterday) - 1
z_t = v_spx[1:]/v_spx[:-1] - 1
# round each return off to 7 decimal places
print('z_t =', z_t.round(7))


z_t = [ 0.0039739 -0.0030005 -0.0018712 -0.0072066  0.0031089  0.0138858
 -0.0028168 -0.0049415 -0.0016079  0.0036985]


From the database db_stocks_sp we upload AON and CVC stock values, and compute portfolio returns

In [8]:
# telling pandas to only use these 3 (out of 430+) columns from this csv file
# use the columns names from row 1
# Load AON and CVC price columns from row 2 onward (row 1 is a title row), using date as index
db_stocks_sp = read_csv("data/db_stocks_sp.csv", usecols=['date', 'AON', 'CVC'],
                     parse_dates=True, index_col=0, header=1, dayfirst=True, date_format='mixed')
# Filter to 16-30 Mar 2012 window; reorder columns to [AON, CVC] (file's raw column order was reversed)
v_stocks = db_stocks_sp[(db_stocks_sp.index >= to_datetime('16-03-2012', dayfirst=True)) &
                        (db_stocks_sp.index <= to_datetime('30-03-2012', dayfirst=True))].values[:, [1,0]]

# returns of 2 stocks
# Daily linear returns for AON (col 0) and CVC (col 1): (price today / price yesterday) - 1
r_n_t = v_stocks[1:, :]/v_stocks[:-1, :] - 1

# Current portfolio returns : [AON, CVC] — must sum to 1
w_tnow = array([0.42268011, 0.57731989])

# Portfolio return each day = weighted sum of AON and CVC returns (eq. 25.6)
x_t = sum(w_tnow*r_n_t, axis=1)

# Display portfolio return series, rounded for readability
print('x_t =', x_t.round(7))

x_t = [ 0.0029476 -0.0066345  0.0015802 -0.0029019  0.0028408  0.0149615
 -0.0095231  0.0031261 -0.0017765 -0.0023666]


Compute Original Regression Loading and Shift

In [10]:
# Sample mean of [portfolio return, S&P return] — estimate of mu*_{X,Z} (eq. 25.11)
mu_xz = average(array([x_t, z_t]), axis=1)
# covariance matrix on [portfolio return, S&P return]
sigma2_xz = cov(array([x_t, z_t]))

# factor loading
# OLS slope: beta_hat = Cov(x,z) / Var(z) — factor loading (eq. 25.9)
# sigma2_xz[0,1] — the covariance between portfolio returns and S&P returns.
# sigma2_xz[1,1] — the variance of S&P returns alone.
beta_reg = sigma2_xz[0,1]/sigma2_xz[1,1]
print('beta_reg =', abs(beta_reg).round(7))

# shift
# mu_xz[0] — mean of x_t (portfolio return).
# mu_xz[1] — mean of z_t (S&P return).
alpha_reg = mu_xz[0] - beta_reg * mu_xz[1]
print('alpha_reg =', abs(alpha_reg).round(7))

# Stress test: predict portfolio return if S&P falls 0.3% (eq. 25.10)
# point statement
# z = -0.003 — a hypothetical scenario: "suppose the S&P drops 0.3% tomorrow" (note: −0.003 as a decimal = −0.3%).
z = -0.003
# point prediction
# x_bar_point = alpha_reg + beta_reg*z
# plugs that hypothetical z into your fitted regression line to predict what the portfolio return would be
# under that scenario. This is exactly equation 25.10's stress test: α̂ + β̂·z.
x_bar_point = alpha_reg + beta_reg*z
print('x_bar_point =', x_bar_point.round(7))


beta_reg = 0.8202328
alpha_reg = 3.9e-05
x_bar_point = -0.0024997


Unearth hidden factors via regression factor scoring (Chapter 26.d Factor Analysis)

In [14]:
# target correlation
# Standardize covariance matrix to correlation matrix: corr = D^-1 Σ D^-1, D = diag(std devs)
cr_xz = (diag(1/sqrt(diag(sigma2_xz))))@sigma2_xz@(diag(1/sqrt(diag(sigma2_xz))))
# MLF fitted model
# Fit ML factor analysis model with 2 latent factors from the correlation matrix
mlf = Factor(n_factor=2, corr=cr_xz, method='ml').fit()
# Re-Scaled Loadings
# Rescale standardized loadings back to original units: beta = D @ loadings
beta = diag(sqrt(diag(sigma2_xz)))@mlf.loadings
# rescaled variance
delta = diag(sigma2_xz) * mlf.uniqueness
print()
print('Delta shape =', delta.shape)
print()
print('Beta shape =', beta.shape)
print()
print('sigma2_xz shape =', sigma2_xz.shape)
print()
print('(z_t.reshape(-1,1) - mu_xz shape', (z_t.reshape(-1,1) - mu_xz).shape)
print()
# factor scores
# Estimate factor score time series via regression scoring formula (using centered data)
h_t_fa = beta/sigma2_xz@(z_t.reshape(-1,1) - mu_xz).T
# Display the second latent factor's estimated score time series
print('h_t_fa \n', h_t_fa[1, :].round(7))


Delta shape = (2,)

Beta shape = (2, 2)

sigma2_xz shape = (2, 2)

(z_t.reshape(-1,1) - mu_xz shape (10, 2)

h_t_fa 
 [-0.6492114  0.5586856  0.3630956  1.2871256 -0.4994043 -2.3658569
  0.526862   0.8948455  0.3174998 -0.6015149]


# Conclusion......

Still to write up
